# Example 6.4.1 / 1: Testing a Policy Constraint

Companion notebook for **6.4.1 Dropping and Restoring Constraints** in the AMPL lab Documentation, section "Testing a Policy Constraint". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Testing a Policy Constraint

Suppose a production model contains a minimum-service rule. In the Python tabs, `ampl` has
already read `model.mod`. The drop of one member is sent with `ampl.eval()`; see the warning
below for why.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;
param demand {PROD} >= 0;
param cost {PROD} >= 0;
var Make {p in PROD} >= 0;
minimize Cost: sum {p in PROD} cost[p] * Make[p];
subject to Service {p in PROD}: Make[p] >= 0.8 * demand[p];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["demand"] = {"bands": 10, "coils": 20, "plate": 5}
ampl.param["cost"] = {"bands": 2, "coils": 3, "plate": 4}

In [5]:
# run.py
ampl.option["solver_msg"] = 0
ampl.solve(solver="highs")
print("all rules:", ampl.obj["Cost"].value())

ampl.eval('drop Service["plate"];')
ampl.solve(solver="highs")
print("without plate:", ampl.obj["Cost"].value(), ampl.var["Make"].to_dict())

ampl.eval('restore Service["plate"];')
ampl.solve(solver="highs")
print("restored:", ampl.obj["Cost"].value())

all rules: 80.0
without plate: 64.0 {'bands': 8, 'coils': 16, 'plate': 0}
restored: 80.0


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PROD := bands coils plate;
param: demand cost :=
  bands 10 2
  coils 20 3
  plate  5 4;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
option solver_msg 0;
solve;
display Cost;
drop Service["plate"];
solve;
display Cost, Make;
restore Service["plate"];
solve;
display Cost;

Cost = 80

Cost = 64

Make [*] :=
bands   8
coils  16
plate   0
;

Cost = 80

